# 02 - Datenqualität und Duplikate prüfem
Der in Notebook 01 inventarisierte TLFS23-Datenbestand wird auf technische Lesbarkeit, Bildformat und identische Inhalte geprüft. Die Rohdaten werden dabei nicht verändert. Der vollständige Audit bildet die Grundlage für die spätere Datenbereinigung.

In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

here = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in (here, *here.parents)
                     if (p / "notebooks").is_dir() and (p / "data").is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Projektordner mit notebooks/ und data/ fehlt.")
if not (PROJECT_ROOT / "src/tlfs23/data.py").is_file():
    raise FileNotFoundError("Den Ordner src/tlfs23 aus dem Paket ins Projekt kopieren.")
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from tlfs23 import data as d
from tlfs23.common import paths, read_csv, read_json, write_json

P = paths(PROJECT_ROOT)
print("Projekt:", PROJECT_ROOT)

inventory = d.inventory(PROJECT_ROOT)
print("Inventarisierte Bilder:", len(inventory))

Projekt: /Users/sharu/Documents/DBU/tamil-finger-sign-mlops
Inventarisierte Bilder: 255158


## 2. Bilder prüfen
Der vollständige Audit dekodiert alle inventarisierten Bilder. Bereits geprüfte, unveränderte Dateien können aus dem Audit-Cache wiederverwendet werden.

In [2]:
AUDIT_MODE = "full"
audit, summary = d.audit(PROJECT_ROOT, mode=AUDIT_MODE, per_class=5)
display(pd.DataFrame([{
    "Modus": summary["audit_mode"],
    "Gepruefte Dateien": summary["n_files_audited"],
    "Nicht lesbar": summary["n_unreadable_files"],
    "Exakte Duplikatgruppen": summary["n_exact_duplicate_groups"],
}]))

Bildpruefung: 500/255158 (500 aus Cache)
Bildpruefung: 1000/255158 (1000 aus Cache)
Bildpruefung: 1500/255158 (1500 aus Cache)
Bildpruefung: 2000/255158 (2000 aus Cache)
Bildpruefung: 2500/255158 (2500 aus Cache)
Bildpruefung: 3000/255158 (3000 aus Cache)
Bildpruefung: 3500/255158 (3500 aus Cache)
Bildpruefung: 4000/255158 (4000 aus Cache)
Bildpruefung: 4500/255158 (4500 aus Cache)
Bildpruefung: 5000/255158 (5000 aus Cache)
Bildpruefung: 5500/255158 (5500 aus Cache)
Bildpruefung: 6000/255158 (6000 aus Cache)
Bildpruefung: 6500/255158 (6500 aus Cache)
Bildpruefung: 7000/255158 (7000 aus Cache)
Bildpruefung: 7500/255158 (7500 aus Cache)
Bildpruefung: 8000/255158 (8000 aus Cache)
Bildpruefung: 8500/255158 (8500 aus Cache)
Bildpruefung: 9000/255158 (9000 aus Cache)
Bildpruefung: 9500/255158 (9500 aus Cache)
Bildpruefung: 10000/255158 (10000 aus Cache)
Bildpruefung: 10500/255158 (10500 aus Cache)
Bildpruefung: 11000/255158 (11000 aus Cache)
Bildpruefung: 11500/255158 (11500 aus Cache)
Bildp

,Modus,Gepruefte Dateien,Nicht lesbar,Exakte Duplikatgruppen
0,full,255158,0,1247


## 3. Formate und Bildqualität
Bilddimensionen, Farbmodus, Helligkeit und Kantenvariation werden beschreibend geprüft. Es werden daraus keine automatischen Ausschlussgrenzen abgeleitet.

In [3]:
display(audit.groupby(["width", "height", "mode"], dropna=False)
        .size().rename("n").reset_index().sort_values("n", ascending=False).head(10))
display(audit[["brightness", "edge_variance"]].describe().round(2))
errors = audit.loc[~audit["readable"], ["relative_path", "error"]]
print("Fehlerhafte Dateien:", len(errors))
display(errors.head(10))

,width,height,mode,n
0,640,480,RGB,255158


,brightness,edge_variance
count,255158.00,255158.00
mean,128.90,1303.28
std,15.61,618.80
min,33.52,188.75
25%,118.05,951.66
50%,130.30,1175.66
75%,140.14,1483.77
max,175.27,6287.72


Fehlerhafte Dateien: 0


,relative_path,error


## 4. Identische Inhalte unterscheiden
Zwei Prüfungen unterscheiden dateiidentische Inhalte und identische dekodierte RGB-Pixel. Wiederkehrende Dateinamen werden separat erfasst und gelten nicht automatisch als Duplikate. Eine Entscheidung über den Ausschluss erfolgt erst im folgenden Kurationsschritt.

In [4]:
exact = audit.loc[audit["sha256_group_size"].gt(1)]
pixel_equal = audit.loc[audit["pixel_sha256_group_size"].gt(1)]
display(pd.DataFrame({
    "Pruefung": ["Dateiidentische Gruppen", "Pixelidentische Gruppen", "Wiederkehrende Dateinamen"],
    "Anzahl": [exact["sha256"].nunique(), pixel_equal["pixel_sha256"].nunique(),
               int(audit["filename"].value_counts().gt(1).sum())],
}))
display(pd.DataFrame({
    "Pruefung": [
        "Exakte Duplikatdateien",
        "Exakte Duplikatgruppen",
        "Ordneruebergreifende exakte Dateien",
        "Pixelidentische Gruppen",
        "Wiederkehrende Dateinamen",
    ],
    "Anzahl": [
        summary["n_exact_duplicate_files"],
        summary["n_exact_duplicate_groups"],
        summary["n_cross_folder_exact_files"],
        pixel_equal["pixel_sha256"].nunique(),
        summary["n_repeated_filenames"],
    ],
}))

,Pruefung,Anzahl
0,Dateiidentische Gruppen,1247
1,Pixelidentische Gruppen,1247
2,Wiederkehrende Dateinamen,1079


,Pruefung,Anzahl
0,Exakte Duplikatdateien,10084
1,Exakte Duplikatgruppen,1247
2,Ordneruebergreifende exakte Dateien,142
3,Pixelidentische Gruppen,1247
4,Wiederkehrende Dateinamen,1079


## 5. Audit abschließen
Nur ein vollständiger Audit wird in Notebook 03 verwendet. Fehlerhafte Dateien und Duplikate werden hier noch nicht entfernt.

In [5]:
print("Audit abgeschlossen.")
print("Geprüfte Dateien:", summary["n_files_audited"])
print("Nicht lesbar:", summary["n_unreadable_files"])

Audit abgeschlossen.
Geprüfte Dateien: 255158
Nicht lesbar: 0


## Ergebnis

Der vollständige Audit umfasst 255.158 Bilder. Alle Dateien konnten erfolgreich dekodiert werden und liegen einheitlich als RGB-Bilder mit 640 × 480 Pixeln vor. Es wurden 10.084 Dateien in 1.247 exakten Duplikatgruppen identifiziert; davon betreffen 142 Dateien ordnerübergreifende Duplikate. Die Rohdaten wurden nicht verändert. Die Entscheidung über den Umgang mit diesen Fällen erfolgt getrennt im Kurationsschritt.